# WP41 Gate A — JupyterLite platform proof

This notebook is infrastructure verification only. It is not student-facing course content. It proves, in one browser notebook, that the platform WP41 builds for every future migrated exercise actually works end to end: a browser Python kernel starts, the common scientific stack imports with no install cell, a same-origin data file loads, two regression models fit and score, a Matplotlib figure renders, a slider/dropdown activity runs, and a checked question grades itself.

In [ ]:
import sys

if sys.platform == 'emscripten':
    # Running in the browser (JupyterLite). ipywidgets has no prebuilt
    # Pyodide package, unlike numpy/pandas/scikit-learn/matplotlib
    # (preloaded by the JupyterLite build itself, see
    # book/lite/overrides.json), so it is fetched here instead of by a
    # student-run install cell. Local Jupyter and Colab already have it.
    import micropip
    await micropip.install(['ipywidgets'])

In [ ]:
# Setup: the common scientific environment. Every package here is
# already available in this notebook -- there is no install cell.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

print('imports ok')

In [ ]:
# Load a small same-origin data asset. No repository access, GitHub
# token, or network fetch is involved: this file is served from the
# same static site as this notebook.
data = pd.read_csv('data/gate_a_sample.csv')
data.head()

In [ ]:
X = data[['feature_1', 'feature_2']].to_numpy()
y = data['target'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=41
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('train:', X_train_scaled.shape, 'test:', X_test_scaled.shape)

In [ ]:
linear_model = LinearRegression().fit(X_train_scaled, y_train)
linear_pred = linear_model.predict(X_test_scaled)

knn_model = KNeighborsRegressor(n_neighbors=5).fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

for name, pred in [('Linear regression', linear_pred), ('KNN (k=5)', knn_pred)]:
    r2 = r2_score(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    print(f'{name}: R2={r2:.3f}  MSE={mse:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(y_test, linear_pred, alpha=0.7)
lims = [min(y_test.min(), linear_pred.min()), max(y_test.max(), linear_pred.max())]
ax.plot(lims, lims, linestyle='--', color='gray', label='Perfect prediction')
ax.set_xlabel('Observed target')
ax.set_ylabel('Predicted target')
ax.set_title('Gate A: observed vs. predicted')
ax.legend()
plt.show()

## Notebook-native interactive: vary k

Move the slider or choose a value from the dropdown to refit KNN at a different k and watch the fit change.

In [ ]:
k_slider = widgets.IntSlider(value=5, min=1, max=25, step=1, description='k (slider):')
k_dropdown = widgets.Dropdown(
    options=[1, 3, 5, 10, 15, 20, 25], value=5, description='k (dropdown):'
)
output = widgets.Output()


def _refit(k):
    model = KNeighborsRegressor(n_neighbors=k).fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    with output:
        output.clear_output(wait=True)
        r2 = r2_score(y_test, pred)
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.scatter(y_test, pred, alpha=0.7)
        ax.set_title(f'k={k}  R2={r2:.3f}')
        ax.set_xlabel('Observed target')
        ax.set_ylabel('Predicted target')
        plt.show()


def _on_slider_change(change):
    if change['name'] == 'value':
        k_dropdown.value = change['new'] if change['new'] in k_dropdown.options else k_dropdown.value
        _refit(change['new'])


def _on_dropdown_change(change):
    if change['name'] == 'value':
        k_slider.value = change['new']


k_slider.observe(_on_slider_change, names='value')
k_dropdown.observe(_on_dropdown_change, names='value')

display(widgets.VBox([k_slider, k_dropdown, output]))
_refit(k_slider.value)

## Checked question

Select every statement that correctly describes this Gate A proof, then press **Check answer**.

In [ ]:
_options = [
    'The scaler was fit on the training predictors only.',
    'The data file was loaded over the public internet with a GitHub token.',
    'Both a linear model and a KNN model were evaluated on held-out test rows.',
]
_correct = {0, 2}

_checkboxes = [widgets.Checkbox(value=False, description=opt, indent=False) for opt in _options]
_check_button = widgets.Button(description='Check answer', button_style='primary')
_feedback = widgets.Output()


def _on_check_clicked(_button):
    selected = {i for i, cb in enumerate(_checkboxes) if cb.value}
    with _feedback:
        _feedback.clear_output(wait=True)
        if selected == _correct:
            print('Correct.')
        else:
            print('Not quite -- try again.')


_check_button.on_click(_on_check_clicked)
display(widgets.VBox([*_checkboxes, _check_button, _feedback]))

## Edit-and-persist check

Automated browser verification edits the token below, reloads the page, downloads the notebook, and confirms the edited token is present in the download.

In [ ]:
# EDIT_TARGET
gate_a_token = 'gate-a-original-token'
print(gate_a_token)